In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
from typing import List, Tuple, Optional
from IPython.display import clear_output
%matplotlib inline

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# desktop version
data_folder = "G:\Meu Drive\mestrado_final_files\data\\"
models_folder = "G:\Meu Drive\mestrado_final_files\models\\"

# google colab version
# data_folder = "drive/MyDrive/mestrado_final_files/data/"
# models_folder = "drive/MyDrive/mestrado_final_files/models/"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [15]:
pd.set_option("display.max_columns", None)

In [16]:
with open(data_folder + 'integer_to_token.pkl', 'rb') as f:
    integer_to_token = pickle.load(f)

df_games_nba = pd.read_csv(data_folder + "games_nba.csv", sep = ';')\
[['GAME_ID', 'HOME_TEAM_ID', 'AWAY_TEAM_ID', 'HOME_TEAM', 'AWAY_TEAM', 'GAME_DATE', 'SEASON']]


In [17]:
BATCH_SIZE = 1024
EPOCHS = 50
CONTEXT_SIZE = 5
VOCAB_SIZE = 601
TIME_SIZE = 301
EMBEDDING_LAYER_SIZE = 32
NUM_HIDDEN_NEURONS = [50, 50, 50]

run_models = 1

In [20]:
full_covariables = ['period1', 'period2', 'period3', 'period4', 'period_overtime', 'time', 'limhfouls', 'limvfouls', 'dif', 'dif_20', 'ind_close_3',
                    'ind_close_5', 'ind_close_8', 'ind_blowout_15', 'ind_last_2_minutes', 'ind_last_minute', 'ind_last_20_seconds', 'ind_last_10_seconds',
                    'lead', 'home_lead', 'away_lead', 'ind_last_2_minutes_game', 'ind_last_minute_game', 'ind_last_20_seconds_game',
                    'ind_last_10_seconds_game', 'ind_clutch_2_last_minutes', 'ind_clutch_2_last_minutes_home', 'ind_clutch_2_last_minutes_away',
                    'ind_clutch_last_minute', 'ind_clutch_last_minute_home', 'ind_clutch_last_minute_away', 'ind_clutch_last_20_seconds',
                    'ind_clutch_last_20_seconds_home', 'ind_clutch_last_20_seconds_away', 'ind_clutch_last_10_seconds', 'ind_clutch_last_10_seconds_home',
                    'ind_clutch_last_10_seconds_away', 'ind_blowout_last_minute', 'ball_possession', 'home_FGM', 'away_FGM', 'home_FGA', 'away_FGA',
                    'home_FGP', 'away_FGP', 'home_3FGM', 'away_3FGM', 'home_3FGA', 'away_3FGA', 'home_3FGP', 'away_3FGP', 'home_FTM', 'away_FTM',
                    'home_FTA', 'away_FTA', 'home_FTP', 'away_FTP', 'home_RebOff', 'away_RebOff', 'home_RebDef', 'away_RebDef', 'home_RebTot',
                    'away_RebTot', 'home_AST', 'away_AST', 'home_STL', 'away_STL', 'home_BLK', 'away_BLK', 'home_TO', 'away_TO', 'home_FP',
                    'away_FP', 'home_points', 'away_points', 'home_PMP', 'away_PMP', 'home_outcome', 'away_outcome', 'playoff_game']

covariables_model = ['period1', 'period2', 'period3', 'period4', 'period_overtime', 'time', 'limhfouls', 'limvfouls', 'dif_20', 'ind_close_3',
                    'ind_close_5', 'ind_close_8', 'ind_blowout_15', 'ind_last_2_minutes', 'ind_last_minute', 'ind_last_20_seconds', 'ind_last_10_seconds',
                    'home_lead', 'away_lead', 'ind_last_2_minutes_game', 'ind_last_minute_game', 'ind_last_20_seconds_game',
                    'ind_last_10_seconds_game', 'ind_clutch_2_last_minutes', 'ind_clutch_2_last_minutes_home', 'ind_clutch_2_last_minutes_away',
                    'ind_clutch_last_minute', 'ind_clutch_last_minute_home', 'ind_clutch_last_minute_away', 'ind_clutch_last_20_seconds',
                    'ind_clutch_last_20_seconds_home', 'ind_clutch_last_20_seconds_away', 'ind_clutch_last_10_seconds', 'ind_clutch_last_10_seconds_home',
                    'ind_clutch_last_10_seconds_away', 'ind_blowout_last_minute', 'ball_possession']

index_covariables = [i for i, var in enumerate(full_covariables) if var in covariables_model]

In [19]:
%run 01_functions_training.ipynb
#%run drive/MyDrive/mestrado_final_files/notebooks/01_functions_training.ipynb

In [21]:
with open(data_folder + 'data_token_features.pkl', 'rb') as f:
    df_token_features = pickle.load(f)

In [22]:
for s in ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23']:
    x1 = df_token_features.query("season == @s & season_split == 'train' & ind_last_2_minutes_game == 0").shape[0]
    x2 = df_token_features.query("season == @s & season_split == 'train' & ind_last_2_minutes_game == 1").shape[0]
    x3 = df_token_features.query("season == @s & season_split == 'dev' & ind_last_2_minutes_game == 0").shape[0]
    x4 = df_token_features.query("season == @s & season_split == 'dev' & ind_last_2_minutes_game == 1").shape[0]
    print(f"Rows train data season {s} model 1: {x1}")
    print(f"Rows dev data season {s} model 1: {x3}")
    print(f"Rows train data season {s} model 2: {x2}")
    print(f"Rows dev data season {s} model 2: {x4}\n")

del x1, x2, x3, x4

Rows train data season 2018-19 model 1: 408457
Rows dev data season 2018-19 model 1: 45601
Rows train data season 2018-19 model 2: 23319
Rows dev data season 2018-19 model 2: 2812

Rows train data season 2019-20 model 1: 356321
Rows dev data season 2019-20 model 1: 39999
Rows train data season 2019-20 model 2: 20721
Rows dev data season 2019-20 model 2: 2327

Rows train data season 2020-21 model 1: 357614
Rows dev data season 2020-21 model 1: 39649
Rows train data season 2020-21 model 2: 20305
Rows dev data season 2020-21 model 2: 2466

Rows train data season 2021-22 model 1: 405196
Rows dev data season 2021-22 model 1: 45152
Rows train data season 2021-22 model 2: 22539
Rows dev data season 2021-22 model 2: 2431

Rows train data season 2022-23 model 1: 402523
Rows dev data season 2022-23 model 1: 44956
Rows train data season 2022-23 model 2: 22931
Rows dev data season 2022-23 model 2: 2511



In [23]:
df_token_features

,game_id,split,season,season_split,period,remaining_time,diff,token_state,ind_last_2_minutes_game,token_features,numerical_features,target_tokens
0,21800001,train,2018-19,train,1,7200.0,0.0,4,0.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",87
1,21800001,train,2018-19,train,1,7200.0,0.0,4,0.0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 87]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",152
2,21800001,train,2018-19,train,1,7000.0,0.0,4,0.0,"[0, 0, 0, 0, 0, 0, 0, 0, 87, 152]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.02777777777777779,...",144
3,21800001,train,2018-19,train,1,6750.0,0.0,4,0.0,"[0, 0, 0, 0, 0, 0, 0, 87, 152, 144]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0625, 0.0, 0.0, 0....",240
4,21800001,train,2018-19,train,1,6730.0,0.0,4,0.0,"[0, 0, 0, 0, 0, 0, 87, 152, 144, 240]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.06527777777777777,...",281
...,...,...,...,...,...,...,...,...,...,...,...,...
2565572,52200211,train,2022-23,train,4,650.0,27.0,161,1.0,"[542, 63, 28, 13, 12, 145, 239, 322, 334, 449]","[0.0, 0.0, 0.0, 1.0, 0.0, 0.9097222222222222, ...",334
2565573,52200211,train,2022-23,train,4,508.0,27.0,170,1.0,"[63, 28, 13, 12, 145, 239, 322, 334, 449, 334]","[0.0, 0.0, 0.0, 1.0, 0.0, 0.9294444444444444, ...",282
2565574,52200211,train,2022-23,train,4,475.0,27.0,170,1.0,"[28, 13, 12, 145, 239, 322, 334, 449, 334, 282]","[0.0, 0.0, 0.0, 1.0, 0.0, 0.9340277777777778, ...",29
2565575,52200211,train,2022-23,train,4,377.0,25.0,170,1.0,"[13, 12, 145, 239, 322, 334, 449, 334, 282, 29]","[0.0, 0.0, 0.0, 1.0, 0.0, 0.9476388888888889, ...",337


In [ ]:
# with open(data_folder + 'data_time_features.pkl', 'rb') as f:
#     df_time_features = pickle.load(f)

In [24]:
trn_tkn_tkn = torch.tensor(df_token_features.query("season_split == 'train' & season == '2018-19' & ind_last_2_minutes_game == 0")["token_features"].to_list(), dtype = torch.int64, device = device)
trn_tkn_num = torch.tensor(df_token_features.query("season_split == 'train' & season == '2018-19' & ind_last_2_minutes_game == 0")["numerical_features"].to_list(), dtype = torch.float32, device = device)
trn_tkn_y   = torch.tensor(df_token_features.query("season_split == 'train' & season == '2018-19' & ind_last_2_minutes_game == 0")["target_tokens"].to_list(), dtype = torch.int64, device = device)

val_tkn_tkn = torch.tensor(df_token_features.query("season_split == 'dev' & season == '2018-19' & ind_last_2_minutes_game == 0")["token_features"].to_list(), dtype = torch.int64, device = device)
val_tkn_num = torch.tensor(df_token_features.query("season_split == 'dev' & season == '2018-19' & ind_last_2_minutes_game == 0")["numerical_features"].to_list(), dtype = torch.float32, device = device)
val_tkn_y   = torch.tensor(df_token_features.query("season_split == 'dev' & season == '2018-19' & ind_last_2_minutes_game == 0")["target_tokens"].to_list(), dtype = torch.int64, device = device)

C:\Users\User\AppData\Local\Temp\ipykernel_14312\949779388.py:2: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  trn_tkn_num = torch.tensor(df_token_features.query("season_split == 'train' & season == '2018-19' & ind_last_2_minutes_game == 0")["numerical_features"].to_list(), dtype = torch.float32, device = device)


In [26]:
token_model = NGramModel_ExtraInfo(context_size = 5, vocab_size = 601, embedding_dim = 32,
                                    numerical_input_dim = len(index_covariables), num_hidden_neurons = [50, 50, 50],
                                    num_layers = 3, activation = nn.GELU, dropout_rate = 0.0, use_layer_norm = False)

token_model, trnloss, devloss = train_num_ngram_model(
    model = token_model,
    train_token_data = trn_tkn_tkn[:, -5:],
    train_numerical_data = trn_tkn_num[:, index_covariables],
    train_targets = trn_tkn_y,
    batch_size = BATCH_SIZE, 
    n_epochs = 50, 
    lr = 0.002,
    val_token_data = val_tkn_tkn[:, -5:],
    val_numerical_data = val_tkn_num[:, index_covariables],
    val_targets = val_tkn_y,
    params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 1})

In [25]:
token_model = NGramModel_ExtraInfo(context_size = 5, vocab_size = 601, embedding_dim = 32,
                                    numerical_input_dim = len(index_covariables), num_hidden_neurons = [50, 50, 50],
                                    num_layers = 3, activation = nn.GELU, dropout_rate = 0.0, use_layer_norm = True)

token_model, trnloss, devloss = train_num_ngram_model(
    model = token_model,
    train_token_data = trn_tkn_tkn[:, -5:],
    train_numerical_data = trn_tkn_num[:, index_covariables],
    train_targets = trn_tkn_y,
    batch_size = BATCH_SIZE, 
    n_epochs = 50, 
    lr = 0.002,
    val_token_data = val_tkn_tkn[:, -5:],
    val_numerical_data = val_tkn_num[:, index_covariables],
    val_targets = val_tkn_y,
    params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 1})

In [29]:
list_seasons = ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23']
list_results = []
filename_mask = "data_{}_season_{}.pkl"
num_shuffles = 5

In [30]:
list_game_ids = df_games_nba.query("SEASON == @season")['GAME_ID'].to_list()
num_games = int(len(list_game_ids) * 0.80)
games_trn = random.sample(list_game_ids, num_games)
games_dev = [game for game in list_game_ids if game not in games_trn]
print(f"Getting the token data for season : {season}")
with open(data_folder + filename_mask.format("token", season.replace("-", "")), 'rb') as f:
    data_tkn = pickle.load(f)

selected_games_tkn_trn = {}
for k in ['token_features', 'numerical_features', 'target_tokens']:
    selected_games_tkn_trn[k] = [item for key, item in data_tkn[k].items() if key in games_trn]
    selected_games_tkn_trn[k] = [sublist for list_ in selected_games_tkn_trn[k] for sublist in list_]
    if k == 'numerical_features':
        selected_games_tkn_trn[k] = torch.tensor(selected_games_tkn_trn[k]).float()
    else:
        selected_games_tkn_trn[k] = torch.tensor(selected_games_tkn_trn[k])

Getting the token data for season : 2021-22


In [37]:
selected_games_tkn_trn.keys()

dict_keys(['token_features', 'numerical_features', 'target_tokens'])

In [38]:
selected_games_tkn_trn['numerical_features'][:10]

tensor([[ 1.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 1.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  1.0000,  1.0000,  1.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000, 

In [15]:

for season in list_seasons:
    dict_model = {}
    list_game_ids = df_games_nba.query("SEASON == @season")['GAME_ID'].to_list()
    num_games = int(len(list_game_ids) * 0.80)
    games_trn = random.sample(list_game_ids, num_games)
    games_dev = [game for game in list_game_ids if game not in games_trn]
    print(f"Getting the token data for season : {season}")
    with open(data_folder + filename_mask.format("token", season.replace("-", "")), 'rb') as f:
        data_tkn = pickle.load(f)

    selected_games_tkn_trn = {}
    for k in ['token_features', 'numerical_features', 'target_tokens']:
        selected_games_tkn_trn[k] = [item for key, item in data_tkn[k].items() if key in games_trn]
        selected_games_tkn_trn[k] = [sublist for list_ in selected_games_tkn_trn[k] for sublist in list_]
        if k == 'numerical_features':
            selected_games_tkn_trn[k] = torch.tensor(selected_games_tkn_trn[k]).float()
        else:
            selected_games_tkn_trn[k] = torch.tensor(selected_games_tkn_trn[k])

    selected_games_tkn_dev = {}
    for k in ['token_features', 'numerical_features', 'target_tokens']:
        selected_games_tkn_dev[k] = [item for key, item in data_tkn[k].items() if key in games_dev]
        selected_games_tkn_dev[k] = [sublist for list_ in selected_games_tkn_dev[k] for sublist in list_]
        if k == 'numerical_features':
            selected_games_tkn_dev[k] = torch.tensor(selected_games_tkn_dev[k]).float()
        else:
            selected_games_tkn_dev[k] = torch.tensor(selected_games_tkn_dev[k])

    constant_value = 1

    if run_models:
        print(f"Training token model for season {season}")
        token_model = NGramModel_ExtraInfo(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, embedding_dim = EMBEDDING_LAYER_SIZE,
                                           numerical_input_dim = len(index_covariables), num_hidden_neurons = NUM_HIDDEN_NEURONS,
                                           num_layers = len(NUM_HIDDEN_NEURONS), activation = nn.GELU, dropout_rate = 0.3)

        token_model, trnloss, devloss = train_num_ngram_model(model = token_model,
                                                              train_token_data = selected_games_tkn_trn['token_features'][:, -CONTEXT_SIZE:],
                                                              train_numerical_data = constant_value * selected_games_tkn_trn['numerical_features'][:, index_covariables],
                                                              train_targets = selected_games_tkn_trn['target_tokens'],
                                                              batch_size = BATCH_SIZE, max_steps = TRAIN_STEPS, lr = 0.002,
                                                              val_token_data = selected_games_tkn_dev['token_features'][:, -CONTEXT_SIZE:],
                                                              val_numerical_data = constant_value * selected_games_tkn_dev['numerical_features'][:, index_covariables],
                                                              val_targets = selected_games_tkn_dev['target_tokens'],
                                                              params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 0})

        print(f"Saving weights of token model for season {season}")
        token_model.eval()
        dict_model["token_model_name"] = "ngram_token_model_season_{}_dropout03_new_variables_parameters.pth".format(season.replace("-", ""))
        dict_model["token_model_config"] = {"embedding_layer_size": EMBEDDING_LAYER_SIZE, "token_context_size": CONTEXT_SIZE, "token_vocab_size": VOCAB_SIZE,
                                            "num_hidden_neurons": NUM_HIDDEN_NEURONS, "batch_size": BATCH_SIZE}
        dict_model["token_model_covariables"] = covariables_model
        dict_model["token_model_train_loss"] = trnloss
        dict_model["token_model_dev_loss"] = devloss

        torch.save(token_model.state_dict(), models_folder + dict_model["token_model_name"])

        total = 0
        for name, param in token_model.named_parameters():
            total += param.numel()

        dict_model["token_model_total_parameters"] = str(total)
        print(f"Total parameters of token model for season {season}: {total}")

        output = token_model(selected_games_tkn_trn['token_features'][:, -CONTEXT_SIZE:], selected_games_tkn_trn['numerical_features'][:, index_covariables])
        baseline_loss = F.cross_entropy(output, selected_games_tkn_trn['target_tokens'])

        dict_model["token_model_traindata_loss"] = float(baseline_loss)
        print(f"Training loss of token model for season {season}: {dict_model['token_model_traindata_loss']}")

        print(f"Getting the variables importance of token model for season {season}")
        df_importance = permutation_importance_ngram(model = token_model, target = selected_games_tkn_dev['target_tokens'],
                                                     tokens = selected_games_tkn_dev['token_features'][:, -CONTEXT_SIZE:],
                                                     numerical_features = selected_games_tkn_dev['numerical_features'][:, index_covariables],
                                                     names_numerical_features = covariables_model, num_repeats = num_shuffles)

        dict_model["token_model_importance_variable"] = df_importance

        print(f"Getting the weights descriptive values of token model for season {season}")
        df_first_layer_weights = get_descriptive_values_weights(token_model, emb_layer_size = EMBEDDING_LAYER_SIZE, context_size = CONTEXT_SIZE,
                                                                covariables = covariables_model)

        dict_model["token_model_weights_describe"] = df_first_layer_weights
        clear_output(wait = True)

    del data_tkn
    print(f"Getting the time data for season : {season}")
    with open(data_folder + filename_mask.format("time", season.replace("-", "")), 'rb') as f:
        data_time = pickle.load(f)

    selected_games_time_trn = {}
    for k in ['token_features', 'numerical_features', 'target_time']:
        selected_games_time_trn[k] = [item for key, item in data_time[k].items() if key in games_trn]
        selected_games_time_trn[k] = [sublist for list_ in selected_games_time_trn[k] for sublist in list_]
        if k == 'numerical_features':
            selected_games_time_trn[k] = torch.tensor(selected_games_time_trn[k]).float()
        else:
            selected_games_time_trn[k] = torch.tensor(selected_games_time_trn[k])

    selected_games_time_dev = {}
    for k in ['token_features', 'numerical_features', 'target_time']:
        selected_games_time_dev[k] = [item for key, item in data_time[k].items() if key in games_dev]
        selected_games_time_dev[k] = [sublist for list_ in selected_games_time_dev[k] for sublist in list_]
        if k == 'numerical_features':
            selected_games_time_dev[k] = torch.tensor(selected_games_time_dev[k]).float()
        else:
            selected_games_time_dev[k] = torch.tensor(selected_games_time_dev[k])

    if run_models:
        print(f"Training time model for season {season}")
        time_model = SpentTimeModel(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, time_size = TIME_SIZE,
                                    embedding_dim = EMBEDDING_LAYER_SIZE, numerical_input_dim = len(covariables_model),
                                    num_hidden_neurons = NUM_HIDDEN_NEURONS, num_layers = len(NUM_HIDDEN_NEURONS),
                                    activation_function = nn.GELU, dropout_rate = 0.3)

        time_model, time_trnlosses, time_devlosses = train_spent_time_model(
            model = time_model,
            train_token_data = selected_games_time_trn['token_features'][:, -CONTEXT_SIZE:],
            train_numerical_data = selected_games_time_trn['numerical_features'][:, index_covariables],
            train_targets = selected_games_time_trn['target_time'],
            batch_size = BATCH_SIZE, max_steps = TRAIN_STEPS, lr =  0.002,
            val_token_data = selected_games_time_dev['token_features'][:, -CONTEXT_SIZE:],
            val_numerical_data =  selected_games_time_dev['numerical_features'][:, index_covariables],
            val_targets =  selected_games_time_dev['target_time'],
            params_optimizer = {'factor': 0.5, 'threshold': 1e-5, 'patience': 0})

        print(f"Saving weights of time model for season {season}")
        time_model.eval()
        dict_model["time_model_name"] = "ngram_time_model_season_{}_dropout03_new_variables_parameters.pth".format(season.replace("-", ""))
        dict_model["time_model_config"] = {"embedding_layer_size": EMBEDDING_LAYER_SIZE, "token_context_size": CONTEXT_SIZE, "token_vocab_size": VOCAB_SIZE,
                                            "num_hidden_neurons": NUM_HIDDEN_NEURONS, "batch_size": BATCH_SIZE}
        dict_model["time_model_covariables"] = covariables_model
        dict_model["time_model_train_loss"] = trnloss
        dict_model["time_model_dev_loss"] = devloss

        torch.save(time_model.state_dict(), models_folder + dict_model["time_model_name"])

        total = 0
        for name, param in time_model.named_parameters():
            total += param.numel()

        dict_model["time_model_total_parameters"] = str(total)
        print(f"Total parameters of time model for season {season}: {total}")

        output = time_model(selected_games_time_trn['token_features'][:, -CONTEXT_SIZE:], selected_games_time_trn['numerical_features'][:, index_covariables])
        baseline_loss = F.cross_entropy(output, selected_games_time_trn['target_time'])

        dict_model["time_model_traindata_loss"] = float(baseline_loss)
        print(f"Training loss of time model for season {season}: {dict_model['time_model_traindata_loss']}")

        print(f"Getting the variables importance of time model for season {season}")
        df_importance = permutation_importance_ngram(model = time_model, target = selected_games_time_dev['target_time'],
                                                     tokens = selected_games_time_dev['token_features'][:, -CONTEXT_SIZE:],
                                                     numerical_features = selected_games_time_dev['numerical_features'][:, index_covariables],
                                                     names_numerical_features = covariables_model, num_repeats = num_shuffles)

        dict_model["time_model_importance_variable"] = df_importance

        print(f"Getting the weights descriptive values of time model for season {season}")
        df_first_layer_weights = get_descriptive_values_weights(time_model, emb_layer_size = EMBEDDING_LAYER_SIZE, context_size = CONTEXT_SIZE,
                                                                covariables = covariables_model)

        dict_model["time_model_weights_describe"] = df_first_layer_weights
        clear_output(wait = True)

    del data_time
    list_results.append(dict_model)
    clear_output(wait = True)


Getting the time data for season : 2022-23
Training time model for season 2022-23
Saving weights of time model for season 2022-23lidation Loss: 2.42410, LR: 6.103515625e-08     
Total parameters of time model for season 2022-23: 49583
Training loss of time model for season 2022-23: 2.4073612689971924
Getting the variables importance of time model for season 2022-23
Baseline model loss:  2.4241
Getting the weights descriptive values of time model for season 2022-23


In [16]:
with open(data_folder + "results_ngram_models_seasons_performance_new_variables.pkl", 'wb') as f:
    pickle.dump(list_results, f)

# old implementation

In [18]:
covariables_model = ['period1', 'period2', 'period3', 'period4', 'period_overtime', 'time', 'limhfouls', 'limvfouls', 'dif', 'ind_last_minute',
                     'ind_last_20_seconds', 'ind_last_10_seconds', 'lead', 'ball_possession']

index_covariables = [i for i, var in enumerate(full_covariables) if var in covariables_model]

In [19]:
season = '2021-22'
list_results = []

In [20]:
dict_model = {}
list_game_ids = df_games_nba.query("SEASON == @season")['GAME_ID'].to_list()
num_games = int(len(list_game_ids) * 0.80)
games_trn = random.sample(list_game_ids, num_games)
games_dev = [game for game in list_game_ids if game not in games_trn]
print(f"Getting the token data for season : {season}")
with open(data_folder + filename_mask.format("token", season.replace("-", "")), 'rb') as f:
    data_tkn = pickle.load(f)

selected_games_tkn_trn = {}
for k in ['token_features', 'numerical_features', 'target_tokens']:
    selected_games_tkn_trn[k] = [item for key, item in data_tkn[k].items() if key in games_trn]
    selected_games_tkn_trn[k] = [sublist for list_ in selected_games_tkn_trn[k] for sublist in list_]
    if k == 'numerical_features':
        selected_games_tkn_trn[k] = torch.tensor(selected_games_tkn_trn[k]).float()
    else:
        selected_games_tkn_trn[k] = torch.tensor(selected_games_tkn_trn[k])

selected_games_tkn_dev = {}
for k in ['token_features', 'numerical_features', 'target_tokens']:
    selected_games_tkn_dev[k] = [item for key, item in data_tkn[k].items() if key in games_dev]
    selected_games_tkn_dev[k] = [sublist for list_ in selected_games_tkn_dev[k] for sublist in list_]
    if k == 'numerical_features':
        selected_games_tkn_dev[k] = torch.tensor(selected_games_tkn_dev[k]).float()
    else:
        selected_games_tkn_dev[k] = torch.tensor(selected_games_tkn_dev[k])

constant_value = 1

if run_models:
    print(f"Training token model for season {season}")
    token_model = NGramModel_ExtraInfo(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, embedding_dim = EMBEDDING_LAYER_SIZE,
                                        numerical_input_dim = len(index_covariables), num_hidden_neurons = NUM_HIDDEN_NEURONS,
                                        num_layers = len(NUM_HIDDEN_NEURONS), activation = nn.GELU, dropout_rate = 0.3)

    token_model, trnloss, devloss = train_num_ngram_model(model = token_model,
                                                            train_token_data = selected_games_tkn_trn['token_features'][:, -CONTEXT_SIZE:],
                                                            train_numerical_data = constant_value * selected_games_tkn_trn['numerical_features'][:, index_covariables],
                                                            train_targets = selected_games_tkn_trn['target_tokens'],
                                                            batch_size = BATCH_SIZE, max_steps = TRAIN_STEPS, lr = 0.002,
                                                            val_token_data = selected_games_tkn_dev['token_features'][:, -CONTEXT_SIZE:],
                                                            val_numerical_data = constant_value * selected_games_tkn_dev['numerical_features'][:, index_covariables],
                                                            val_targets = selected_games_tkn_dev['target_tokens'],
                                                            params_optimizer = {'factor': 0.5, 'threshold': 0, 'patience': 0})

    print(f"Saving weights of token model for season {season}")
    token_model.eval()
    dict_model["token_model_name"] = "ngram_token_model_season_{}_dropout03_new_variables_parameters.pth".format(season.replace("-", ""))
    dict_model["token_model_config"] = {"embedding_layer_size": EMBEDDING_LAYER_SIZE, "token_context_size": CONTEXT_SIZE, "token_vocab_size": VOCAB_SIZE,
                                        "num_hidden_neurons": NUM_HIDDEN_NEURONS, "batch_size": BATCH_SIZE}
    dict_model["token_model_covariables"] = covariables_model
    dict_model["token_model_train_loss"] = trnloss
    dict_model["token_model_dev_loss"] = devloss

    torch.save(token_model.state_dict(), models_folder + dict_model["token_model_name"])

    total = 0
    for name, param in token_model.named_parameters():
        total += param.numel()

    dict_model["token_model_total_parameters"] = str(total)
    print(f"Total parameters of token model for season {season}: {total}")

    output = token_model(selected_games_tkn_trn['token_features'][:, -CONTEXT_SIZE:], selected_games_tkn_trn['numerical_features'][:, index_covariables])
    baseline_loss = F.cross_entropy(output, selected_games_tkn_trn['target_tokens'])

    dict_model["token_model_traindata_loss"] = float(baseline_loss)
    print(f"Training loss of token model for season {season}: {dict_model['token_model_traindata_loss']}")

    print(f"Getting the variables importance of token model for season {season}")
    df_importance = permutation_importance_ngram(model = token_model, target = selected_games_tkn_dev['target_tokens'],
                                                    tokens = selected_games_tkn_dev['token_features'][:, -CONTEXT_SIZE:],
                                                    numerical_features = selected_games_tkn_dev['numerical_features'][:, index_covariables],
                                                    names_numerical_features = covariables_model, num_repeats = num_shuffles)

    dict_model["token_model_importance_variable"] = df_importance

    print(f"Getting the weights descriptive values of token model for season {season}")
    df_first_layer_weights = get_descriptive_values_weights(token_model, emb_layer_size = EMBEDDING_LAYER_SIZE, context_size = CONTEXT_SIZE,
                                                            covariables = covariables_model)

    dict_model["token_model_weights_describe"] = df_first_layer_weights
    clear_output(wait = True)

del data_tkn
print(f"Getting the time data for season : {season}")
with open(data_folder + filename_mask.format("time", season.replace("-", "")), 'rb') as f:
    data_time = pickle.load(f)

selected_games_time_trn = {}
for k in ['token_features', 'numerical_features', 'target_time']:
    selected_games_time_trn[k] = [item for key, item in data_time[k].items() if key in games_trn]
    selected_games_time_trn[k] = [sublist for list_ in selected_games_time_trn[k] for sublist in list_]
    if k == 'numerical_features':
        selected_games_time_trn[k] = torch.tensor(selected_games_time_trn[k]).float()
    else:
        selected_games_time_trn[k] = torch.tensor(selected_games_time_trn[k])

selected_games_time_dev = {}
for k in ['token_features', 'numerical_features', 'target_time']:
    selected_games_time_dev[k] = [item for key, item in data_time[k].items() if key in games_dev]
    selected_games_time_dev[k] = [sublist for list_ in selected_games_time_dev[k] for sublist in list_]
    if k == 'numerical_features':
        selected_games_time_dev[k] = torch.tensor(selected_games_time_dev[k]).float()
    else:
        selected_games_time_dev[k] = torch.tensor(selected_games_time_dev[k])

if run_models:
    print(f"Training time model for season {season}")
    time_model = SpentTimeModel(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, time_size = TIME_SIZE,
                                embedding_dim = EMBEDDING_LAYER_SIZE, numerical_input_dim = len(covariables_model),
                                num_hidden_neurons = NUM_HIDDEN_NEURONS, num_layers = len(NUM_HIDDEN_NEURONS),
                                activation_function = nn.GELU, dropout_rate = 0.3)

    time_model, time_trnlosses, time_devlosses = train_spent_time_model(
        model = time_model,
        train_token_data = selected_games_time_trn['token_features'][:, -CONTEXT_SIZE:],
        train_numerical_data = selected_games_time_trn['numerical_features'][:, index_covariables],
        train_targets = selected_games_time_trn['target_time'],
        batch_size = BATCH_SIZE, max_steps = TRAIN_STEPS, lr =  0.002,
        val_token_data = selected_games_time_dev['token_features'][:, -CONTEXT_SIZE:],
        val_numerical_data =  selected_games_time_dev['numerical_features'][:, index_covariables],
        val_targets =  selected_games_time_dev['target_time'],
        params_optimizer = {'factor': 0.5, 'threshold': 1e-5, 'patience': 0})

    print(f"Saving weights of time model for season {season}")
    time_model.eval()
    dict_model["time_model_name"] = "ngram_time_model_season_{}_dropout03_new_variables_parameters.pth".format(season.replace("-", ""))
    dict_model["time_model_config"] = {"embedding_layer_size": EMBEDDING_LAYER_SIZE, "token_context_size": CONTEXT_SIZE, "token_vocab_size": VOCAB_SIZE,
                                        "num_hidden_neurons": NUM_HIDDEN_NEURONS, "batch_size": BATCH_SIZE}
    dict_model["time_model_covariables"] = covariables_model
    dict_model["time_model_train_loss"] = trnloss
    dict_model["time_model_dev_loss"] = devloss

    torch.save(time_model.state_dict(), models_folder + dict_model["time_model_name"])

    total = 0
    for name, param in time_model.named_parameters():
        total += param.numel()

    dict_model["time_model_total_parameters"] = str(total)
    print(f"Total parameters of time model for season {season}: {total}")

    output = time_model(selected_games_time_trn['token_features'][:, -CONTEXT_SIZE:], selected_games_time_trn['numerical_features'][:, index_covariables])
    baseline_loss = F.cross_entropy(output, selected_games_time_trn['target_time'])

    dict_model["time_model_traindata_loss"] = float(baseline_loss)
    print(f"Training loss of time model for season {season}: {dict_model['time_model_traindata_loss']}")

    print(f"Getting the variables importance of time model for season {season}")
    df_importance = permutation_importance_ngram(model = time_model, target = selected_games_time_dev['target_time'],
                                                    tokens = selected_games_time_dev['token_features'][:, -CONTEXT_SIZE:],
                                                    numerical_features = selected_games_time_dev['numerical_features'][:, index_covariables],
                                                    names_numerical_features = covariables_model, num_repeats = num_shuffles)

    dict_model["time_model_importance_variable"] = df_importance

    print(f"Getting the weights descriptive values of time model for season {season}")
    df_first_layer_weights = get_descriptive_values_weights(time_model, emb_layer_size = EMBEDDING_LAYER_SIZE, context_size = CONTEXT_SIZE,
                                                            covariables = covariables_model)

    dict_model["time_model_weights_describe"] = df_first_layer_weights
    clear_output(wait = True)

del data_time
list_results.append(dict_model)


Getting the time data for season : 2021-22
Training time model for season 2021-22
Saving weights of time model for season 2021-22lidation Loss: 2.43497, LR: 6.103515625e-08     
Total parameters of time model for season 2021-22: 48433
Training loss of time model for season 2021-22: 2.4165544509887695
Getting the variables importance of time model for season 2021-22
Baseline model loss:  2.4350
Getting the weights descriptive values of time model for season 2021-22


In [21]:
list_results

[{'token_model_name': 'ngram_token_model_season_202122_dropout03_new_variables_parameters.pth',
  'token_model_config': {'embedding_layer_size': 32,
   'token_context_size': 5,
   'token_vocab_size': 601,
   'num_hidden_neurons': [50, 50, 50],
   'batch_size': 1024},
  'token_model_covariables': ['period1',
   'period2',
   'period3',
   'period4',
   'period_overtime',
   'time',
   'limhfouls',
   'limvfouls',
   'dif',
   'ind_last_minute',
   'ind_last_20_seconds',
   'ind_last_10_seconds',
   'lead',
   'ball_possession'],
  'token_model_train_loss': [6.404473304748535,
   6.395249843597412,
   6.377873420715332,
   6.36845064163208,
   6.351774215698242,
   6.330312728881836,
   6.29962682723999,
   6.285317420959473,
   6.2414231300354,
   6.198920249938965,
   6.132533073425293,
   6.049746990203857,
   5.999471664428711,
   5.882613182067871,
   5.784640789031982,
   5.656997203826904,
   5.5204973220825195,
   5.261287212371826,
   5.358128547668457,
   5.166971206665039,
   